<a href="https://colab.research.google.com/github/Yami2912/HER360/blob/feature%2Fhealthcare/05_model_decision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
import os, pathlib, sys, json
TOKEN = userdata.get("GITHUB_TOKEN")
os.chdir("/content")
if not os.path.exists("/content/HER360"):
    !git clone -b feature/healthcare https://{TOKEN}@github.com/Yami2912/HER360.git
%cd /content/HER360
!git config user.name  "Yami2912"
!git config user.email "yamipatel0618@gmail.com"
!git fetch origin
!git checkout feature/healthcare
!git pull origin feature/healthcare

BASE = pathlib.Path("/content/HER360/HER360-Healthcare-ML")
print(BASE.exists(), (BASE / "datasets/fertility/ovulation_predictor/processed/ovulation_model_table_v1.csv").exists())

In [ ]:
%%writefile /content/HER360/HER360-Healthcare-ML/models/fertility/ovulation_predictor/ovulation_estimators.py
"""
HER360 Fertility - Ovulation Predictor: estimator definitions (v1)

Kept in a module (not in a notebook) so that:
  - every notebook uses identical definitions,
  - a saved model can be loaded later by the API.
"""
import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor


class GlobalMedian(BaseEstimator, RegressorMixin):
    """Always predict the training median."""
    def fit(self, X, y):
        self.m_ = float(np.median(y))
        return self

    def predict(self, X):
        return np.full(len(X), self.m_)


class CalendarOffset(BaseEstimator, RegressorMixin):
    """prev_cycle_mean + offset, where offset is learned from the training data."""
    def fit(self, X, y):
        self.offset_ = float(np.median(np.asarray(y) - X["prev_cycle_mean"].to_numpy()))
        return self

    def predict(self, X):
        return X["prev_cycle_mean"].to_numpy() + self.offset_


def _imputer():
    return SimpleImputer(strategy="median", add_indicator=True)


def build_models():
    """The five models compared in notebook 04 (same settings)."""
    return {
        "global_median": GlobalMedian(),
        "calendar_offset": CalendarOffset(),
        "linear_regression": make_pipeline(_imputer(), LinearRegression()),
        "random_forest": make_pipeline(_imputer(), RandomForestRegressor(
            n_estimators=300, min_samples_leaf=10, max_features=0.7,
            random_state=42, n_jobs=-1)),
        "gradient_boosting": make_pipeline(_imputer(), GradientBoostingRegressor(
            n_estimators=150, learning_rate=0.03, max_depth=2,
            subsample=0.8, min_samples_leaf=10, random_state=42)),
    }

In [ ]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import GroupKFold, cross_val_predict

sys.path.insert(0, str(BASE / "preprocessing/fertility"))
sys.path.insert(0, str(BASE / "models/fertility/ovulation_predictor"))
import ovulation_prep as op
import ovulation_estimators as oe

T, G, F = op.TARGET, op.GROUP, op.FEATURES_V1
df = pd.read_csv(BASE / "datasets/fertility/ovulation_predictor/processed/ovulation_model_table_v1.csv")
tr = df[(df.split == "train")      & (df.n_prev_cycles >= 1)].reset_index(drop=True)
va = df[(df.split == "validation") & (df.n_prev_cycles >= 1)].reset_index(drop=True)
print("train rows:", len(tr), "| validation rows:", len(va), "| validation participants:", va[G].nunique())

models = oe.build_models()
gkf = GroupKFold(n_splits=5)

# absolute error of every row, out-of-fold, for every model (train split)
oof = {n: cross_val_predict(m, tr[F], tr[T], cv=gkf, groups=tr[G]) for n, m in models.items()}
err_cv = pd.DataFrame({n: np.abs(tr[T] - p) for n, p in oof.items()})
err_cv[G] = tr[G]

# absolute error on validation, after fitting on all training rows
for m in models.values():
    m.fit(tr[F], tr[T])
err_val = pd.DataFrame({n: np.abs(va[T] - m.predict(va[F])) for n, m in models.items()})
err_val[G] = va[G]
print("done")

In [ ]:
def paired_gain(err, cand, ref="calendar_offset", n_boot=5000, seed=42):
    """Average error reduction of `cand` vs `ref`, with a participant-level bootstrap 95% range."""
    d = (err[ref] - err[cand]).groupby(err[G]).agg(["mean", "size"])
    gains, w = d["mean"].to_numpy(), d["size"].to_numpy()
    rng = np.random.default_rng(seed)
    k = len(d)
    boots = [np.average(gains[i], weights=w[i]) for i in (rng.integers(0, k, k) for _ in range(n_boot))]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"gain": round(float(np.average(gains, weights=w)), 3),
            "ci_low": round(float(lo), 3), "ci_high": round(float(hi), 3), "participants": int(k)}

rows = []
for cand in ["linear_regression", "random_forest", "gradient_boosting"]:
    cv, v = paired_gain(err_cv, cand), paired_gain(err_val, cand)
    rows.append({"model": cand,
                 "cv_gain": cv["gain"], "cv_ci_low": cv["ci_low"], "cv_ci_high": cv["ci_high"],
                 "val_gain": v["gain"], "val_participants": v["participants"]})
gain_table = pd.DataFrame(rows)
display(gain_table)

In [ ]:
MIN_GAIN_DAYS = 0.10          # the rule's minimum average gain (days)

gain_table["passes_rule"] = (
    (gain_table.cv_ci_low > 0)
    & (gain_table.cv_gain >= MIN_GAIN_DAYS)
    & (gain_table.val_gain >= 0)
)
display(gain_table)

winners = gain_table[gain_table.passes_rule].sort_values("cv_gain", ascending=False)
chosen = str(winners.model.iloc[0]) if len(winners) else "calendar_offset"
print("\nVERDICT for v1:", chosen)

NB_DIR = BASE / "notebooks/fertility/ovulation_predictor"
gain_table.to_csv(NB_DIR / "model_decision_table_v1.csv", index=False)
(NB_DIR / "model_decision_v1.json").write_text(json.dumps({
    "chosen_v1_approach": chosen,
    "rule": f"ML replaces calendar_offset only if CV gain 95% range excludes 0, mean CV gain >= {MIN_GAIN_DAYS:.2f} days, and validation gain >= 0",
    "method": "participant-level bootstrap (5000 resamples) of paired absolute-error differences vs calendar_offset",
    "cv": "GroupKFold(5), train split, rows with >=1 previous cycle",
    "test_split_used": False,
    "note": "Engineering decision for a college project. Not medical guidance.",
}, indent=2))
print("saved")

In [ ]:
%cd /content/HER360
!git add HER360-Healthcare-ML/models HER360-Healthcare-ML/notebooks
!git commit -m "Fertility: estimator module and v1 model decision"
!git push origin feature/healthcare